# invsqrt (fusion): phasing by 1/sqrt(x)

Builds and checks only this task, writing it to the `submission/` folder.
The same code runs inside the merged notebook `vqcc_submission.ipynb`,
which builds all three tasks and writes `submission.zip`.

In [ ]:
# ---- Settings ---------------------------------------------------------------
VERIFY = 'quick'      # 'none' | 'quick' (minutes) | 'full' (official verifiers, ~30-60 min)
USE_IMPROVED = {'invsqrt': True, 'dqi': True, 'dfthcsos': False}
REQUIRE_FRONTIER = True   # stop with an error if the frontier circuits can't be built (instead of a much lower score)

### Optional: run locally

On Kaggle, leave this as `None`: the data is found with `kagglehub`.
To run on your own machine, set `LOCAL_DATA` to the folder where you unpacked the
competition data (the folder that contains `verifiers/` and `sample_submission/`,
or any folder above it).

In [ ]:
LOCAL_DATA = None     # e.g. 'data/'

## 1. Setup

Finds the competition data, imports kickmix and the official verifiers, and
prepares the output folder. Data is looked up in this order:

1. `LOCAL_DATA` (the optional cell above), or the `VQCC_DATA` environment variable;
2. `kagglehub.competition_download('google-verifiable-quantum-circuit-compilation')`;
3. any copy mounted under `/kaggle/input`.

kickmix is imported and checked: if the import picks up the data's `kickmix/` folder
instead of the real module (the cause of the "no attribute CircuitBuilder" errors),
the compiled `kickmix*.so` from the data is loaded directly.

Two public datasets are downloaded with `kagglehub.dataset_download`:
`ahoirg/qcc-reproducible-circuit-assets` (frontier circuits, section 1b) and
`marculera/dfthc-compact-pipeline` (chemistry factorization pipeline).

In [ ]:
import glob
import io
import itertools
import json
import math
import os
import shutil
import sys
import time
import zipfile
from dataclasses import dataclass
from fractions import Fraction
from pathlib import Path

import numpy as np


COMPETITION = 'google-verifiable-quantum-circuit-compilation'
MARKER = Path('verifiers') / 'invsqrt' / 'verify_submission_invsqrt.py'


def _data_root_under(folder) -> Path | None:
    """The folder holding verifiers/ (the given folder or one below it), if any."""
    folder = Path(folder).expanduser().resolve()
    if (folder / MARKER).exists():
        return folder
    for hit in sorted(folder.glob('**/' + MARKER.as_posix())):
        return hit.parents[2]
    return None


def find_data_root() -> Path:
    tried = []
    for folder in (LOCAL_DATA, os.environ.get('VQCC_DATA')):
        if folder:
            tried.append(str(folder))
            root = _data_root_under(folder) if Path(folder).expanduser().exists() else None
            if root:
                return root
    try:
        import kagglehub
        path = kagglehub.competition_download(COMPETITION)
        print('kagglehub:', path)
        tried.append(f'kagglehub ({path})')
        root = _data_root_under(path)
        if root:
            return root
    except Exception as ex:
        tried.append(f'kagglehub failed: {ex}')
    if Path('/kaggle/input').exists():
        tried.append('/kaggle/input')
        root = _data_root_under('/kaggle/input')
        if root:
            return root
    raise FileNotFoundError('Competition data not found. Tried: ' + '; '.join(tried) +
                            '. On Kaggle, attach the competition data; locally, set LOCAL_DATA.')


KICKMIX_API = ('CircuitBuilder', 'Circuit', 'GF2Field')


def _is_kickmix(module):
    return all(hasattr(module, name) for name in KICKMIX_API)


def _find_files(root, pattern, max_depth=6):
    """Files under root matching pattern, following symlinked folders (Path.glob('**') does not)."""
    import fnmatch
    root, hits = Path(root), []
    if not root.is_dir():
        return hits
    base = len(root.parts)
    for folder, dirs, files in os.walk(root, followlinks=True):
        if len(Path(folder).parts) - base >= max_depth:
            dirs[:] = []
        hits += [Path(folder) / f for f in files if fnmatch.fnmatch(f, pattern)]
    return sorted(hits)


def load_kickmix(data_root):
    """Return the real kickmix module.

    A plain `import kickmix` can pick up a *folder* named kickmix (such as the one in
    the competition data) as an empty namespace package, which then has no
    CircuitBuilder. So the imported module is checked, and if it is not the real one,
    the compiled kickmix*.so shipped with the data is loaded from its file path.
    """
    import importlib
    import importlib.util
    tried = []
    try:
        import kickmix
        if _is_kickmix(kickmix):
            return kickmix
        where = getattr(kickmix, '__file__', None) or list(getattr(kickmix, '__path__', []))
        tried.append(f'import kickmix gave {where} without {KICKMIX_API}')
    except ImportError as ex:
        tried.append(f'import kickmix: {ex}')
    sys.modules.pop('kickmix', None)
    roots = [Path(data_root)] + ([Path('/kaggle/input')] if Path('/kaggle/input').exists() else [])
    for so in [p for r in roots for p in _find_files(r, 'kickmix*.so')]:
        try:
            spec = importlib.util.spec_from_file_location('kickmix', so)
            module = importlib.util.module_from_spec(spec)
            sys.modules['kickmix'] = module
            spec.loader.exec_module(module)
            if _is_kickmix(module):
                return module
            tried.append(f'{so}: loaded but lacks {KICKMIX_API}')
        except Exception as ex:
            tried.append(f'{so}: {type(ex).__name__}: {ex}')
        sys.modules.pop('kickmix', None)
    for wheel in [p for r in roots for p in _find_files(r, 'kickmix*.whl')]:
        import subprocess
        res = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', str(wheel)],
                             capture_output=True, text=True)
        importlib.invalidate_caches()
        try:
            import kickmix
            if _is_kickmix(kickmix):
                return kickmix
        except ImportError:
            pass
        sys.modules.pop('kickmix', None)
        tried.append(f'{wheel}: pip install -> {res.returncode} {res.stderr.strip()[-200:]}')
    folder = Path(data_root) / 'kickmix'
    listing = sorted(p.name for p in folder.iterdir()) if folder.is_dir() else 'missing'
    raise ImportError(f'kickmix not usable (Python {sys.version.split()[0]}). Tried: ' + '; '.join(tried) +
                      f'. Contents of {folder}: {listing}')


def download_dataset(handle):
    """kagglehub.dataset_download(handle), or None (with the reason printed)."""
    try:
        import kagglehub
        path = kagglehub.dataset_download(handle)
        print(f'{handle}: {path}')
        return Path(path)
    except Exception as ex:
        print(f'{handle}: kagglehub download failed ({type(ex).__name__}: {ex})')
        return None


DATA = find_data_root()
VERIFIERS = DATA / 'verifiers'
SAMPLE = DATA / 'sample_submission'
km = load_kickmix(DATA)
sys.path.insert(0, str(VERIFIERS))

# Public datasets: frontier circuit assets (section 1b) and the chemistry pipeline.
ASSETS_PATH = download_dataset('ahoirg/qcc-reproducible-circuit-assets')
CHEM_PIPELINE_PATH = download_dataset('marculera/dfthc-compact-pipeline')

OUT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('output').resolve()
SUB = OUT / 'submission'
if SUB.exists():
    shutil.rmtree(SUB)
(SUB / 'dfthcsos').mkdir(parents=True)
REPORT = {}
print('data:', DATA)
print('kickmix:', km.__file__, '| version', getattr(km, '__version__', '?'), '| Python', sys.version.split()[0])
print('output:', SUB)

## 1b. Public frontier components (credited)

The strongest public circuits for all three tasks are built by many Kaggle
participants and shared under Apache-2.0. We start from them, verify every one
with the official verifiers, and keep our own circuits (sections 2-4) as the
fallback. They come from **ahoirg's [QCC Reproducible Circuit Assets](https://www.kaggle.com/datasets/ahoirg/qcc-reproducible-circuit-assets), release 2**
(downloaded with `kagglehub.dataset_download` in the setup cell):

| Task | Component | Credits (see the dataset's CREDITS files for the full lineage) |
|---|---|---|
| invsqrt | 62-qubit circuit with five proved rewrites | mrmeows / team selenium (architecture), busyaprime, yoyobar, Ahmed Berat Özer, Ehan Mallick, SomeJoint; rewrite matcher by Kirill Tymchishin; lineage heliosli, Phuc Thai, Dmitrii Gluzdov, marsyas |
| dqi | 891-qubit decoder | SomeJoint / Agniv Banerjee and the team credited in their source (Ahmed Berat Özer, Alex Chilton, jday96314, mrmeows, manwithacat, Dmitrii Gluzdov, ViberTeam, Phuc Thai), yoyobar, ahoirg |
| dfthcsos | C39 factorization + paired-product loaders | marculera (factorization pipeline, tensorqrom), Syed Asad Ali / SomeJoint (C39 loaders), mrmeows, heliosli, Ahmed Berat Özer, kozykappa, Michael Rustad and others |

**Newer public release (preferred when attached):** heliosli's
[qcc-circuits-live](https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy) dataset,
portfolio `v812e4edf89e_y33_ay3` (local GM 24.94): yoyobar's version-33 62-qubit invsqrt; their
update of yoyobar's version-38 892-qubit decoder (lineage kirilltym, daifanhao, syed3000, SomeJoint,
ahoirg, Atakan Aldemir and the earlier decoder line); syed3000's (8, 143, 44) chemistry table and
angle circuits with weights refined from maoxin123's refit (lineage marculera, Ahmed Berat Özer,
heliosli and others). Each file is used only if its SHA-256 matches the pinned value below.

What this notebook runs from the ahoirg dataset (all hash-checked against the
dataset's manifest first):

- `inverse_factor.py`: an exact five-window gate rewrite of the invsqrt circuit,
  each edit proved on all 128 inputs of its 7 wires.
- `scheduled_emit.py` and `tensorqrom.py`: emit the 15 chemistry loaders from the
  C39 angle table and its algebraic plans.
- `v2-dqi.kmb.xz`: the prebuilt dqi decoder, decompressed.

In [ ]:
USE_FRONTIER = {'invsqrt': True, 'dqi': True, 'dfthcsos': True}
LOCAL_ASSETS = None          # local folder holding qcc-reproducible-circuit-assets(-v2); found automatically on Kaggle
ASSETS_DATASET = 'ahoirg/qcc-reproducible-circuit-assets'


def _sha256(path):
    import hashlib
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 24), b''):
            h.update(chunk)
    return h.hexdigest()


def find_assets():
    """Folder holding release 2 of the assets (v2-assets-manifest.json), or None."""
    roots = [Path(p) for p in (LOCAL_ASSETS, os.environ.get('VQCC_ASSETS'), ASSETS_PATH) if p]
    roots.append(Path('/kaggle/input'))
    for root in roots:
        if root.exists():
            for hit in sorted(root.expanduser().glob('**/v2-assets-manifest.json')):
                return hit.parent
    return None


def load_frontier():
    """Authenticate the assets, extract the source bundle and build the frontier
    invsqrt and chemistry circuits. Returns a dict of paths, or raises."""
    assets = find_assets()
    if assets is None:
        raise FileNotFoundError(f'attach the {ASSETS_DATASET} dataset (release 2)')
    manifest = json.loads((assets / 'v2-assets-manifest.json').read_text())
    for name, pin in manifest['files'].items():
        if (assets / name).exists() and _sha256(assets / name) != pin['sha256']:
            raise ValueError(f'{name}: SHA-256 differs from the assets manifest')
    work = OUT / 'frontier_build'
    if work.exists():
        shutil.rmtree(work)
    src = work / 'source'
    src.mkdir(parents=True)
    import hashlib
    bundle = json.loads((assets / 'v2-source-bundle.json').read_text())
    for name, entry in bundle['files'].items():
        if hashlib.sha256(entry['text'].encode()).hexdigest() != entry['sha256']:
            raise ValueError(f'source bundle member {name} fails its hash')
        (src / name).parent.mkdir(parents=True, exist_ok=True)
        (src / name).write_text(entry['text'])
    sys.path.insert(0, str(src))
    from inverse_factor import factor_windows, prove_edits
    from scheduled_emit import build_circuit
    pins = json.loads((src / 'manifest.json').read_text())
    # invsqrt: base circuit -> five proved rewrites
    base = assets / 'v2-inverse-control.kmb'
    text = work / 'inverse_control.kmx'
    km.Circuit.from_file(str(base), format='kmb').to_file(str(text), format='kmx')
    lines, edits = factor_windows(text.read_text().splitlines())
    proof = prove_edits(edits)
    inv_path = work / 'invsqrt.kmb'
    tmp = work / 'inverse_transformed.kmx'
    tmp.write_text('\n'.join(lines) + '\n')
    km.Circuit.from_file(str(tmp), format='kmx').to_file(str(inv_path), format='kmb')
    # chemistry: 15 loaders from the C39 table and its plans
    chem = work / 'dfthcsos'
    chem.mkdir()
    factor = assets / 'v2-c39-factor.npz'
    with np.load(factor, allow_pickle=False) as z:
        words = z['angle_words']
    for slot in range(15):
        plan = json.loads((src / 'plans' / f'slot-{slot}.json').read_text())
        cur = words[:, slot * 5:slot * 5 + 5]
        prev = words[:, (slot - 1) * 5:slot * 5] if slot else np.zeros_like(cur)
        delta = np.vstack([cur ^ prev, np.asarray(plan['unused_address_completion'], dtype=np.int64)])
        terms = [tuple(int(v, 16) for v in t) for t in plan['terms']]
        build_circuit(delta, terms, plan['low_cache_schedule']).to_file(str(chem / f'angle_k_{slot}.kmb'), format='kmb')
    shutil.copyfile(factor, chem / 'submission.npz')
    pinned = {**{k: v['sha256'] for k, v in pins['output_members'].items()}}
    mism = [k for k, h in pinned.items() if (work / k).exists() and _sha256(work / k) != h]
    if mism:
        raise ValueError(f'rebuilt files differ from the pinned outputs: {mism}')
    # dqi: decompress the prebuilt decoder
    import lzma
    dqi_path = work / 'dqi.kmb'
    with lzma.open(assets / 'v2-dqi.kmb.xz') as fin, open(dqi_path, 'wb') as fout:
        shutil.copyfileobj(fin, fout, 1 << 24)
    if _sha256(dqi_path) != manifest['build_inputs']['dqi']['sha256']:
        raise ValueError('decompressed dqi.kmb fails its hash')
    print('frontier built:', dict(proof=proof, files_checked=len(pinned) + 1))
    return dict(invsqrt=inv_path, dqi=dqi_path, dfthcsos=chem)


# Newer public release (heliosli/qcc-circuits-live, portfolio v812e4edf89e_y33_ay3, local GM 24.94):
# prebuilt files only, each pinned here by SHA-256. Used per file when present and exact; otherwise the
# ahoirg build above is kept.
PUBLIC_RELEASE_DATASET = 'heliosli/qcc-circuits-live'
PUBLIC_RELEASE_PORTFOLIO = 'v812e4edf89e_y33_ay3'
PUBLIC_RELEASE_SHA256 = {
    'invsqrt.kmb': 'c17971aff984830dc68f4f1459cdc549fd2637dc656bc7d3d70aeabc7c8eb775',
    'dqi.kmb': '4de6b023fbf146d29cc608fd08c15ce12c298f5771fd0d8a678efc403929d212',
    'dfthcsos/submission.npz': '388b6a7ee7d882e9e8d165a79812c3ecb6c7e2486de59fd63733ad82cf61baac',
    'dfthcsos/angle_k_0.kmb': 'd3aab7479fe3b8510e3af3797a14ec8d9bfa226d7aa7717ae4e93f496d2d9538',
    'dfthcsos/angle_k_1.kmb': '4595e5dcc982295db2520c8497219e1768757edf3aa9e706fa44b1716f996ed2',
    'dfthcsos/angle_k_2.kmb': '77dec28c5ebffe723fb58717a31c017d66c44cbec220bc3cc5a15a2fff051aa3',
    'dfthcsos/angle_k_3.kmb': '8e137a014a1325450eaf9ec15bc824475b757b3a05bbc542eed9cccfd6b5ec53',
    'dfthcsos/angle_k_4.kmb': 'd9f2a0d739142a1d20789352c617cccc702a8f0b4b17f91971e4e4ef3e6f6c3f',
    'dfthcsos/angle_k_5.kmb': 'bd0886bb6c3d5c04f5e1fd29dbd2562d1c7e5b1a7406bc1abde6819093a0880e',
    'dfthcsos/angle_k_6.kmb': '12c0c6f92d708ff7d852f2c6f36a40ecf4369249fa1f0ab4527b08e0152e7558',
    'dfthcsos/angle_k_7.kmb': '93dc9742392a0dd6e754aebbbea1519014292a56b6908dfce5ae417f2cde73d6',
    'dfthcsos/angle_k_8.kmb': 'd43bd96b88e6ef38c7ea9e697856646af4259889b75d1bc67a94a1768670b5a6',
    'dfthcsos/angle_k_9.kmb': 'a5960a572c39a3996b3919eb8bc18066b8aa78d3fc83bc7435d7e5550de3a2a6',
    'dfthcsos/angle_k_10.kmb': 'f28f2ead2b7eb3d45aadf4e0bb9f9399c7d8f016247d7020ae39dbd9c062cace',
    'dfthcsos/angle_k_11.kmb': '7b9094ddc215c6c7db223fe319e6d2a6f67c1669f412e4d3281c280723df5b81',
    'dfthcsos/angle_k_12.kmb': 'd49dfbfdb8cff3c3d680e4ef65a3d1a01db7a2fc666f49a522d7f58f50305e78',
    'dfthcsos/angle_k_13.kmb': 'cb7e48a7029323a6ec11c0f6dede5f5e0443f67da92e0e6554ff9120b8279efd',
    'dfthcsos/angle_k_14.kmb': '07fcdcb75d3a9e7f2573e057376ec20b7a56bf9a53fe7d7f00fb351614c2003f',
}
LOCAL_RELEASE = None   # local folder holding QCC_RELEASE_MANIFEST.json; found automatically on Kaggle


def load_public_release():
    """Copy the pinned public-release files that are present and exact. Returns {task: path}."""
    import lzma
    roots = [Path(p) for p in (LOCAL_RELEASE, os.environ.get('QCC_DS_DIR')) if p]
    got = download_dataset(PUBLIC_RELEASE_DATASET)
    if got:
        roots.append(got)
    roots.append(Path('/kaggle/input'))
    manifest_path = next((m for r in roots if r.exists() for m in _find_files(r, 'QCC_RELEASE_MANIFEST.json')), None)
    if manifest_path is None:
        print(f'{PUBLIC_RELEASE_DATASET}: not attached; keeping the ahoirg build')
        return {}
    ds = manifest_path.parent
    manifest = json.loads(manifest_path.read_text())
    if manifest.get('portfolio') != PUBLIC_RELEASE_PORTFOLIO:
        print(f"{PUBLIC_RELEASE_DATASET}: portfolio {manifest.get('portfolio')} is not the pinned "
              f'{PUBLIC_RELEASE_PORTFOLIO}; only files with pinned hashes are used')
    out = OUT / 'public_release'
    if out.exists():
        shutil.rmtree(out)
    exact = set()
    for item in manifest['files']:
        name = item['file']
        if name not in PUBLIC_RELEASE_SHA256:
            continue
        src, dst = ds / item['stored_as'], out / name
        if not src.exists():
            continue
        dst.parent.mkdir(parents=True, exist_ok=True)
        if item['stored_as'].endswith('.xz'):
            with lzma.open(src) as fi, open(dst, 'wb') as fo:
                shutil.copyfileobj(fi, fo, 1 << 22)
        else:
            shutil.copyfile(src, dst)
        if _sha256(dst) == PUBLIC_RELEASE_SHA256[name]:
            exact.add(name)
        else:
            dst.unlink()
    found = {}
    if 'invsqrt.kmb' in exact:
        found['invsqrt'] = out / 'invsqrt.kmb'
    if 'dqi.kmb' in exact:
        found['dqi'] = out / 'dqi.kmb'
    chem = [n for n in PUBLIC_RELEASE_SHA256 if n.startswith('dfthcsos/')]
    if all(n in exact for n in chem):
        found['dfthcsos'] = out / 'dfthcsos'
    print(f'{PUBLIC_RELEASE_DATASET}: exact files for {sorted(found)}')
    return found


FRONTIER = {}
if any(USE_FRONTIER.values()):
    problems = []
    try:
        FRONTIER = load_frontier()
    except Exception as ex:
        problems.append(f'ahoirg build: {type(ex).__name__}: {ex}')
    try:
        FRONTIER.update(load_public_release())
    except Exception as ex:
        problems.append(f'{PUBLIC_RELEASE_DATASET}: {type(ex).__name__}: {ex}')
    missing = [t for t, on in USE_FRONTIER.items() if on and t not in FRONTIER]
    if missing:
        msg = (f'frontier circuits missing for {missing}, so the submission would score far lower. '
               f'Attach {ASSETS_DATASET} and/or {PUBLIC_RELEASE_DATASET}, or set REQUIRE_FRONTIER = False. '
               f'Causes: {problems}')
        if REQUIRE_FRONTIER:
            raise RuntimeError(msg)
        print(msg)
print('frontier sources:', {k: str(v) for k, v in FRONTIER.items()})

## 2. invsqrt: phasing by 1/sqrt(x)

**Contract.** A 22-qubit register `x` holds 1 <= x <= 3,133,452. The circuit
must apply the phase exp(2 pi i / sqrt(x)) with relative error at most
2^-24 on every input, and return all scratch qubits to 0.
Cost = qubits x (Toffolis + T/4); rotations are priced in T gates.

**Baseline** (260 qubits, 7,735 Toffolis): cubic first guess, one Newton step,
then a full reverse pass.

**Our design.**

1. *Normalize* with a 5-stage binary search on leading zeros (shifts of 16, 8,
   4, 2, 1). Then x_norm = x * 2^t has its top bit set, M = x_norm / 2^21 is
   in [1, 2), and 1/sqrt(x) = p * 2^(sh - 10) with sh = t >> 1 and
   p = M^-1/2 (times 1/sqrt(2) when t is even), so p is in (1/2, 1].
   The stage flags *are* the binary digits of t, so no conversion is needed.
   Clearing each flag afterwards uses measurement plus a phase fix that only
   runs on half the shots.
2. *Look up* the coefficients of a quadratic for M's segment
   (address = parity of t and the top k bits of M). kickmix's lookup costs
   about one Toffoli per entry and clearing it costs about a tenth of that.
3. *Evaluate* y = c0 - (c1 - c2*d)*d with truncated fixed-point multipliers,
   where d is the rest of M's bits.
   - The multipliers use radix-4 Booth recoding: each row handles two bits of d
     with a digit in {-2, ..., 2}, so there are half as many carry-propagating
     subtractions and half as much truncation noise. A negative digit costs
     nothing extra: complement the target with CNOTs, subtract, complement back.
   - Each row is subtracted only from the low bits that can change, plus a
     borrow qubit, followed by one controlled decrement of the high bits.
   - Row bits are cleared by X-basis measurement with conditioned CZ fixes, so
     clearing them costs no Toffolis.
4. *Shift* the phased part of y left by sh with kickmix's barrel shifter, apply
   one Z_POW per bit, then undo steps 4 to 1.

Coefficients are fitted per segment, then refined by least squares against the
exact truncated arithmetic, so the model below reproduces the circuit's
output bit for bit on all 3.1M inputs.

In [ ]:
INV_WIDTH, INV_MAX_X = 22, 3 * (2**10 - 2)**2


@dataclass(frozen=True)
class InvDesign:
    k: int        # mantissa bits in the lookup address (table has 2^(k+1) entries)
    frac: tuple   # fractional bits of registers c0 (= output y), c1, ..., c_deg
    dbits: tuple  # delta bits used by the Horner step that produces c0, c1, ...
    pf: int       # bits of p below the binary point that get phased

    @property
    def degree(self):
        return len(self.frac) - 1

    @property
    def widths(self):
        return [f + (1 if j == 0 else 0) for j, f in enumerate(self.frac)]


def inv_rows(d, j):
    """(x_norm bit index, weight exponent m) for each multiplier row of step j."""
    top = INV_WIDTH - 2 - d.k
    return [(i, 21 - i) for i in range(top, max(top - d.dbits[j], -1), -1)]


def inv_inputs():
    xs = np.arange(1, INV_MAX_X + 1, dtype=np.int64)
    lz = INV_WIDTH - 1 - np.floor(np.log2(xs)).astype(np.int64)
    xn = xs << lz
    assert ((xn >> (INV_WIDTH - 1)) == 1).all()
    e = lz & 1
    return xs, lz, xn, e


def inv_booth_digits(d, j):
    """Radix-4 Booth digits of step j: (lo, mid, hi, m) with x_norm bit indices
    (None = constant 0); digit = -2*hi + mid + lo, weight 2^-m."""
    used = sorted(i for i, _ in inv_rows(d, j))
    n = len(used)
    bit = lambda t: used[t] if 0 <= t < n else None
    m0 = 21 - used[0]
    out = []
    for jj in range((n + 2) // 2):
        lo, mid, hi = bit(2 * jj - 1), bit(2 * jj), bit(2 * jj + 1)
        if (lo, mid, hi) != (None, None, None):
            out.append((lo, mid, hi, m0 - 2 * jj))
    return out


def _inv_shift(r, s):
    return (r >> s) if s >= 0 else (r << -s)


def inv_steps(d, table, xn, e):
    """Exact fixed-point Horner evaluation. Yields (j, c_j, result_j) per step."""
    addr = (e << d.k) | ((xn >> (INV_WIDTH - 1 - d.k)) & ((1 << d.k) - 1))
    r = table[addr, d.degree].copy()
    for j in range(d.degree - 1, -1, -1):
        acc = table[addr, j].copy()
        fin, fout = d.frac[j + 1], d.frac[j]
        for lo, mid, hi, m in inv_booth_digits(d, j):
            g = lambda i: ((xn >> i) & 1) if i is not None else 0
            dg = -2 * g(hi) + g(mid) + g(lo)
            s = m + fin - fout
            row = np.where(np.abs(dg) == 2, _inv_shift(r, s - 1), np.where(np.abs(dg) == 1, _inv_shift(r, s), 0))
            acc -= np.sign(dg) * row
        yield j, table[addr, j], acc
        r = acc


def inv_evaluate(d, table, xn, e):
    """Returns y (the final Horner result) in units of 2^-frac[0]."""
    for j, c, acc in inv_steps(d, table, xn, e):
        pass
    return acc


def inv_phase_turns(d, y, lz):
    """The phase the circuit applies, in turns (bits below 2^-(pf+10) are not phased)."""
    return (((y << (lz >> 1)) >> (d.frac[0] - d.pf))) / 2.0**(d.pf + 10)


def inv_fit(d, refine_iters=3):
    """Per-segment polynomial fit, then least-squares refinement on the exact arithmetic."""
    xs, lz, xn, e = inv_inputs()
    nseg, h = 1 << d.k, 2.0**-d.k
    table = np.zeros((2 * nseg, d.degree + 1), dtype=np.int64)
    nodes = np.cos(np.pi * (np.arange(4 * d.degree + 8) + 0.5) / (4 * d.degree + 8)) * 0.5 + 0.5
    for ee in (0, 1):
        scale = 1.0 if ee else 1 / math.sqrt(2)
        for s in range(nseg):
            c = np.polynomial.polynomial.polyfit(nodes * h, scale / np.sqrt(1 + s * h + nodes * h), d.degree)
            table[(ee << d.k) | s] = [round(abs(c[j]) * 2**d.frac[j]) for j in range(d.degree + 1)]
    addr = (e << d.k) | ((xn >> (INV_WIDTH - 1 - d.k)) & (nseg - 1))
    dl = (xn & ((1 << (INV_WIDTH - 1 - d.k)) - 1)) / 2.0**21
    target = 1 / np.sqrt(xs.astype(np.float64))
    scale = 2.0**(10 - (lz >> 1))
    order = np.argsort(addr, kind='stable')
    bounds = np.searchsorted(addr[order], np.arange(2 * nseg + 1))
    for it in range(refine_iters + 1):
        err = (inv_phase_turns(d, inv_evaluate(d, table, xn, e), lz) - target) * scale
        for a in range(2 * nseg):
            idx = order[bounds[a]:bounds[a + 1]]
            if len(idx) == 0:
                continue
            if it < refine_iters:
                coef = np.linalg.lstsq(np.vander(dl[idx], d.degree + 1, increasing=True), err[idx], rcond=None)[0]
                for j in range(d.degree + 1):
                    table[a, j] -= round((1 if j % 2 == 0 else -1) * coef[j] * 2**d.frac[j])
            else:
                rel = err[idx] / (target[idx] * scale[idx])
                table[a, 0] -= round((rel.max() + rel.min()) / 2 * (target[idx] * scale[idx]).mean() * 2**d.frac[0])
    y = inv_evaluate(d, table, xn, e)
    phase = inv_phase_turns(d, y, lz)
    stats = dict(
        worst_ratio=float((np.abs(phase - target) / (2.0**-24 * target)).max()),
        min_margin_turns=float((2.0**-24 * target - np.abs(phase - target)).min()),
        low_widths=inv_low_widths(d, table, xn, e),
    )
    return table, y, stats


def inv_low_widths(d, table, xn, e):
    """Bits needed for the total amount P each Horner step subtracts (over all x).
    The borrow trick needs 0 <= P < 2^w and a non-negative result."""
    out = [0] * d.degree
    for j, c, acc in inv_steps(d, table, xn, e):
        P = c - acc
        assert P.min() >= 0 and (acc >= 0).all(), 'Horner range check failed'
        out[j] = int(P.max()).bit_length()
    return out

### Circuit builder

In [ ]:
INV_STAGES = (16, 8, 4, 2, 1)   # normalization shifts; flag L means "top L bits of x are 0"


def _inv_flag_zero_top(b, x, L, flag):
    top = x[INV_WIDTH - L:]
    b.x(top)
    if L == 1:
        b.cx(top[0], flag)
    else:
        b.multi_controlled_x(top, flag)
    b.x(top)


def _inv_mbu_clear_flag(b, x, L, flag):
    """Measure flag in the X basis; on outcome 1, apply (-1)^flag computed from x."""
    m = b.alloc_bits(1)[0]
    b.hmr(flag, m)
    top = x[INV_WIDTH - L:]
    b.x(top)
    b.push_condition(m)
    if L == 1:
        b.z(top[0])
    elif L == 2:
        b.cz(top[0], top[1])
    else:
        anc = b.alloc_qubits(L - 2)
        b.init_and(top[0], top[1], anc[0])
        for i in range(1, L - 2):
            b.init_and(anc[i - 1], top[i + 1], anc[i])
        b.cz(anc[L - 3], top[L - 1])
        for i in range(L - 3, 0, -1):
            b.del_and(anc[i - 1], top[i + 1], anc[i])
        b.del_and(top[0], top[1], anc[0])
        b.free(anc)
    b.pop_condition()
    b.x(top)
    b.bit_store(m, False)
    b.free(m)


def inv_normalize(b, x, flags, inverse):
    stages = list(zip(INV_STAGES, flags))
    for L, f in (reversed(stages) if inverse else stages):
        with b.mark(f'{"un" if inverse else ""}norm_{L}'):
            if not inverse:
                _inv_flag_zero_top(b, x, L, f)
                for j in range(INV_WIDTH - 1, L - 1, -1):
                    b.cswap(f, x[j], x[j - L])
            else:
                for j in range(L, INV_WIDTH):
                    b.cswap(f, x[j], x[j - L])
                _inv_mbu_clear_flag(b, x, L, f)


def _inv_booth_row(b, inverse, x, lo, mid, hi, r, s, tgt, scratch, wtop):
    """tgt -= digit * (r >> s) (or += when inverse), digit = -2*hi + mid + lo from x bits.

    Row bit p = one * r[p+s] xor two * r[p+s-1], where one = mid xor lo and
    two = (not one) and (hi xor mid). Bits at or above len(tgt) are dropped:
    tgt is mod-2^len(tgt) arithmetic and the final value is known to fit.
    """
    g = lambda i: x[i] if i is not None else None
    qlo, qmid, qhi = g(lo), g(mid), g(hi)
    R = lambda t: r[t] if 0 <= t < len(r) else None
    o, t = b.alloc_qubits(2)
    for q in (qmid, qlo):                       # o = one
        if q is not None:
            b.cx(q, o)
    both = qhi is not None and qmid is not None
    if both:
        b.cx(qmid, qhi)                         # hi temporarily holds hi xor mid
    hm = qhi if qhi is not None else qmid
    has_two = hm is not None
    if has_two:                                 # t = two
        b.x(o); b.init_and(o, hm, t); b.x(o)
    pos1 = [p for p in range(wtop + 1) if R(p + s) is not None]
    pos2 = [p for p in range(wtop + 1) if R(p + s - 1) is not None] if has_two else []
    positions = sorted(set(pos1) | set(pos2))
    if positions:
        for p in pos1:
            b.init_and(o, R(p + s), scratch[p])
        for p in pos2:
            if p in pos1:
                b.ccx(t, R(p + s - 1), scratch[p])
            else:
                b.init_and(t, R(p + s - 1), scratch[p])
        if both:
            b.cx(qmid, qhi)                     # restore hi: it is the digit's sign
        span = tgt[positions[0]:]
        if qhi is not None:
            b.cx(qhi, span)                     # negative digit: ~(~t - v) = t + v
        if inverse:
            b.iadd(scratch[positions[0]:len(tgt)], target=span)
        else:
            b.isub(scratch[positions[0]:len(tgt)], target=span)
        if qhi is not None:
            b.cx(qhi, span)
        if both:
            b.cx(qmid, qhi)
        mb = b.alloc_bits(len(positions))       # clear row bits by measurement
        for p, m in zip(positions, mb):
            b.hmr(scratch[p], m)
            b.push_condition(m)
            if p in pos1:
                b.cz(o, R(p + s))
            if p in pos2:
                b.cz(t, R(p + s - 1))
            b.pop_condition()
        b.bit_store(mb, False)
        b.free(mb)
    if has_two:
        b.x(o); b.del_and(o, hm, t); b.x(o)
    if both:
        b.cx(qmid, qhi)
    for q in (qlo, qmid):
        if q is not None:
            b.cx(q, o)
    b.free(km.array([o, t]))


def inv_horner(b, d, x, regs, scratch, borrows, low_widths, inverse):
    steps = list(range(d.degree - 1, -1, -1))
    for j in (reversed(steps) if inverse else steps):
        acc, r = regs[j], regs[j + 1]
        fin, fout, wl = d.frac[j + 1], d.frac[j], low_widths[j]
        tgt = km.array(list(acc[:wl]) + [borrows[j]])
        with b.mark(f'horner_{j}'):
            if inverse:
                b.iadd(1, target=acc[wl:], control=borrows[j])
            digits = inv_booth_digits(d, j)
            for lo, mid, hi, m in (reversed(digits) if inverse else digits):
                _inv_booth_row(b, inverse, x, lo, mid, hi, r, m + fin - fout, tgt, scratch, wl)
            if not inverse:
                b.isub(1, target=acc[wl:], control=borrows[j])


def inv_build(d, table, low_widths, compute_only=False, max_qubits=None):
    b = km.CircuitBuilder()
    if max_qubits:
        b.set_max_qubits(max_qubits)   # kickmix adders then use fewer carry qubits
    x = b.create_quantum_register(INV_WIDTH, name='x')
    alloc = (lambda n, name: b.create_quantum_register(n, name=name)) if compute_only else (lambda n, name: b.alloc_qubits(n))
    flags = alloc(len(INV_STAGES), 'flags')          # flags[4] (shift 1) is the parity of t
    widths = d.widths
    coef = alloc(sum(widths), 'coef')
    offs = [sum(widths[:j]) for j in range(len(widths))]
    regs = [coef[offs[j]:offs[j] + widths[j]] for j in range(len(widths))]
    ext = alloc(10, 'ext')
    borrows = alloc(d.degree, 'borrows')
    scratch = alloc(max(low_widths) + 1, 'scratch')
    words = []
    for row in table:
        v = 0
        for j, c in enumerate(row):
            assert 0 <= int(c) < (1 << widths[j]), 'coefficient does not fit its register'
            v |= int(c) << offs[j]
        words.append(v)
    address = km.array([x[INV_WIDTH - 1 - d.k + i] for i in range(d.k)] + [flags[4]])
    sh = km.array([flags[3], flags[2], flags[1], flags[0]])         # sh = t >> 1, little-endian
    yh = km.array(list(regs[0][d.frac[0] - d.pf:]) + list(ext))     # phased part of y, LSB = 2^-pf

    def compute(inverse):
        if not inverse:
            inv_normalize(b, x, flags, False)
            with b.mark('lookup'):
                b.init_lookup(table=words, address=address, target=coef)
            inv_horner(b, d, x, regs, scratch, borrows, low_widths, False)
            with b.mark('denormalize'):
                b.left_rotate(yh, sh)
        else:
            with b.mark('renormalize'):
                b.right_rotate(yh, sh)
            inv_horner(b, d, x, regs, scratch, borrows, low_widths, True)
            with b.mark('unlookup'):
                b.del_lookup(table=words, address=address, target=coef)
            inv_normalize(b, x, flags, True)

    with b.mark('compute'):
        compute(False)
    if compute_only:
        return b.finish_circuit()
    with b.mark('phase'):
        for pos in range(len(yh)):
            w = pos - d.pf - 10                       # this bit is worth 2^w turns
            if w < 0:                                 # whole turns do nothing
                b.z_pow(yh[pos], Fraction(2) ** (w + 1))
    with b.mark('uncompute'):
        compute(True)
    b.free(scratch); b.free(borrows); b.free(ext); b.free(coef); b.free(flags)
    return b.finish_circuit()

### Build and check

Three levels of checks, each stricter than the last:

1. **Model, all inputs.** The exact fixed-point model must stay within 97% of
   the tolerance on every x (the remaining margin pays for rotation synthesis).
2. **Circuit = model.** Simulate the compute half of the circuit on 4,096
   inputs (all small x, the largest x and random ones) and require the
   shifted output register to match the model bit for bit.
3. **Official verifier.** `quick`: the first 65,536 inputs, 4 shots each.
   `full`: every input, 64 shots each, exactly as the leaderboard runs it.

In [ ]:
# Found by a local search over widths, scored by the real circuit's qubits x Toffolis.
INV_DESIGN = InvDesign(k=8, frac=(27, 19, 9), dbits=(13, 14), pf=27)
INV_MAX_QUBITS = 128


def inv_check_circuit_matches_model(d, table, y, low_widths):
    xs, lz, xn, e = inv_inputs()
    c = inv_build(d, table, low_widths, compute_only=True)
    rng = np.random.default_rng(0)
    sample = np.unique(np.concatenate([np.arange(1, 2049), rng.integers(1, INV_MAX_X + 1, 2047), [INV_MAX_X]]))
    sim = km.Simulator(batch_size=len(sample))
    sim.use_same_registers_as(c)
    sim.write_across_shots(c.register_data['x'], (sample[None, :] >> np.arange(22)[:, None]) & 1 == 1)
    sim.do(c)
    rd = lambda name: sim.read_across_shots(c.register_data[name]).astype(np.int64)
    coef, ext = rd('coef'), rd('ext')
    g = d.frac[0] - d.pf
    yh_bits = np.concatenate([coef[g:d.widths[0]], ext])
    yh = (yh_bits << np.arange(len(yh_bits))[:, None]).sum(0)
    low = (coef[:g] << np.arange(g)[:, None]).sum(0)
    ys, sh = y[sample - 1], lz[sample - 1] >> 1
    return bool(np.array_equal(yh, (ys >> g) << sh) and np.array_equal(low, ys & ((1 << g) - 1)))


def inv_estimated_score(circuit, stats, mean_toffolis):
    n_rot = INV_DESIGN.pf + 10 - 3
    eps = math.pi * stats['min_margin_turns'] / n_rot
    t = 1 + n_rot * (0.56 * math.log2(1 / eps) + 5.3)
    return circuit.num_qubits * (mean_toffolis + t / 4)


def run_invsqrt():
    from invsqrt import verify_submission_invsqrt as V
    t0 = time.time()
    table, y, stats = inv_fit(INV_DESIGN)
    print(f"model: worst error {stats['worst_ratio']:.3f} x tolerance on all {INV_MAX_X:,} inputs")
    if stats['worst_ratio'] > 0.97:
        raise AssertionError('model error too close to tolerance')
    circuit = inv_build(INV_DESIGN, table, stats['low_widths'], max_qubits=INV_MAX_QUBITS)
    if not inv_check_circuit_matches_model(INV_DESIGN, table, y, stats['low_widths']):
        raise AssertionError('circuit output differs from the model')
    print('circuit matches model bit for bit on 4,096 sample inputs')
    path = SUB / 'invsqrt.kmb'
    circuit.to_file(str(path), format='kmb')
    result = dict(num_qubits=circuit.num_qubits,
                  static_toffolis=circuit.operation_statistics().num_toffoli_like)
    if VERIFY in ('quick', 'full'):
        full = VERIFY == 'full'
        spec = V.ProblemSpecInvSqrt.for_n_p(10) if full else V.ProblemSpecInvSqrt(
            width=22, max_x=65536, beta=Fraction(1), precision_bits=24, shots_per_x=4, system_qubits=0)
        r = V.verify(km.Circuit.from_file(str(path)), spec)
        result.update(verifier='full' if full else 'quick (x <= 65,536)', verifier_status=r['status'],
                      mean_toffolis=r['toffoli_count'])
        result['score'] = r['score'] if full else inv_estimated_score(circuit, stats, r['toffoli_count'])
    else:
        result['score'] = inv_estimated_score(circuit, stats, result['static_toffolis'])
    result['seconds'] = round(time.time() - t0, 1)
    return result


def use_baseline_invsqrt():
    src = SAMPLE / 'invsqrt.kmb'
    if src.exists():
        shutil.copy(src, SUB / 'invsqrt.kmb')
    else:
        from invsqrt.nr_reference_circuit import build_nr_circuit
        build_nr_circuit(10).to_file(str(SUB / 'invsqrt.kmb'), format='kmb')


def run_frontier_invsqrt():
    from invsqrt import verify_submission_invsqrt as V
    t0 = time.time()
    shutil.copy(FRONTIER['invsqrt'], SUB / 'invsqrt.kmb')
    c = km.Circuit.from_file(str(SUB / 'invsqrt.kmb'))
    if VERIFY == 'none':
        return dict(num_qubits=c.num_qubits, static_toffolis=c.operation_statistics().num_toffoli_like,
                    verifier='none (hash-pinned build)', seconds=round(time.time() - t0, 1))
    full = VERIFY == 'full'
    spec = V.ProblemSpecInvSqrt.for_n_p(10) if full else V.ProblemSpecInvSqrt(
        width=22, max_x=65536, beta=Fraction(1), precision_bits=24, shots_per_x=4, system_qubits=0)
    r = V.verify(c, spec)
    return dict(num_qubits=c.num_qubits, mean_toffolis=r['toffoli_count'], rotations=r['rotation_count'],
                verifier='full' if full else 'quick (x <= 65,536; score estimate)', verifier_status=r['status'],
                score=r['score'], seconds=round(time.time() - t0, 1))


REPORT['invsqrt'] = None
if USE_FRONTIER['invsqrt'] and 'invsqrt' in FRONTIER:
    try:
        REPORT['invsqrt'] = dict(version='frontier', **run_frontier_invsqrt())
    except Exception as ex:
        print('invsqrt: frontier circuit rejected:', ex)
if REPORT['invsqrt'] is None and USE_IMPROVED['invsqrt']:
    try:
        REPORT['invsqrt'] = dict(version='ours', **run_invsqrt())
    except Exception as ex:
        print('invsqrt: our circuit rejected:', ex)
if REPORT['invsqrt'] is None:
    use_baseline_invsqrt()
    REPORT['invsqrt'] = dict(version='baseline')
print(json.dumps(REPORT['invsqrt'], indent=2))

## Summary

In [ ]:
print(json.dumps(REPORT, indent=2, default=str))
for f in sorted(SUB.rglob('*')):
    if f.is_file():
        print(f'{f.relative_to(SUB)}  {f.stat().st_size / 1e6:.1f} MB')